# RAG Experiments Notebook

An experimentation notebook (not a final pipeline): each cell does one thing, with an explanation above it.
The full Docling + bge-m3 + Qdrant + Groq stack, plus the pieces that were missing from the course 5.25 notebook.

| Part | What it does |
|---|---|
| 1 | Setup: imports and a single settings cell |
| 2 | Ingestion: Docling → chunks → bge-m3 → Qdrant (no rebuild on every run) |
| 3 | `@step`: caching + time + tokens per step |
| 4 | Search: vector / BM25 / hybrid (with optional filter) + reranker |
| 5 | Quick comparison of the four methods |
| 6 | LLM steps: decompose / rewrite / multi-query / HyDE / **self-query** + generate |
| 7 | `Config`, `retrieve`, `ask` and the ladder (6 stages) |
| 8 | Golden set |
| 9 | Run the ladder on all questions (results are saved) |
| 10 | Retrieval evaluation with **ranx** |
| 11 | Answer evaluation with **Ragas** |
| 12 | Time and tokens per step, and the scorecard |
| 13 | Usage: refusal gate, `chat_ask`, `find` |

# Part 1: Setup

## 1.1 Imports and settings

All settings live in a single cell. `REBUILD = False` means the collection is not deleted on every run (set it to True only after changing the chunking).
`WORKERS` is the number of questions processed in parallel; lower it if Groq returns a rate limit.

In [ ]:
# pip install -q docling docling-core transformers qdrant-client langchain-core langchain-huggingface langchain-groq rank-bm25 sentence-transformers ranx "ragas==0.4.3" openai pandas tqdm

import asyncio, getpass, json, os, re, time, uuid
from concurrent.futures import ThreadPoolExecutor
from contextlib import nullcontext
from contextvars import ContextVar
from dataclasses import dataclass, replace
from functools import cache, wraps
from pathlib import Path
from threading import Lock

import pandas as pd
from tqdm import tqdm 

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["RAGAS_DO_NOT_TRACK"] = "true"
pd.set_option("display.width", 200, "display.max_colwidth", 80)

# ── Settings ──
DATA_DIR      = Path("data/")
GOLDEN_PATH   = "data/golden_fina.csv"
EXPERIMENTS   = Path("experiments")
EXPERIMENTS.mkdir(exist_ok=True)

QDRANT_URL    = "http://localhost:6333"
TABLE         = "advanced_chunks"
DIM           = 1024
EMBED_MODEL   = "BAAI/bge-m3"
RERANK_MODEL  = "BAAI/bge-reranker-v2-m3"
LLM_MODEL     = "openai/gpt-oss-120b"     # Groq
LLM_MODEL_JUDGE ="gemini-2.5-flash-lite  "      # "gemini-2.5-flash"

REBUILD       = False   # True = delete the collection and rebuild from scratch
WORKERS       = 2       # how many questions run in parallel (Groq rate limit)

# Part 2: Ingestion

## 2.1 Docling: settings, converter and chunker

Maximum chunk size is 1024 tokens (using the bge-m3 tokenizer), any chunk smaller than 250 is merged with the next one, and trivial fragments (under 30 tokens) are dropped.

In [ ]:
from transformers import AutoTokenizer
from langchain_core.documents import Document

from docling.chunking import HybridChunker
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import HeadingHierarchyOptions, PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling_core.types.doc.document import DoclingDocument, SectionHeaderItem
from docling_core.types.doc.labels import DocItemLabel

CACHE_DIR = Path("docling_cache/")
CACHE_DIR.mkdir(exist_ok=True)

MAX_TOKENS = 1024   # max chunk size from Docling
MIN_TOKENS = 250    # any chunk smaller than this is merged with the next one
HARD_MAX   = 1200   # merging never exceeds this number
DROP_BELOW = 30     # trivial fragments (e.g. "O'REILLY®") are dropped

# First content page per file (optional). If a file isn't listed, it is detected automatically.  Example: {"RAG for Production": 25}
FIRST_PAGE_OVERRIDE = {}

# headings that get misparsed and are removed from the heading path
NOISE_HEADINGS = {"note", "tip", "warning", "caution", "important",
                  "constant width", "constant width italic", "italic", "bold"}
# front/back matter sections are removed entirely
NOISE_SECTIONS = re.compile(
    r"praise for|revision history|how to contact|online learning|conventions used|"
    r"table of contents|^contents$|copyright|about the authors?|colophon|"
    r"فهرس المحتويات|حقوق الطبع",
    re.I,
)
# start of the actual content (used to auto-detect the first page)
CONTENT_START = re.compile(
    r"^(chapter\s+\d+[\.:]\s|part\s+[ivx\d]+[\.:]\s|\d+\.?\s+introduction|introduction$|abstract$|"
    r"الفصل\s|الباب\s|مقدمة|ملخص)",
    re.I,
)

pipeline_options = PdfPipelineOptions()
pipeline_options.generate_parsed_pages = True  # for style hints in headings
pipeline_options.heading_hierarchy_options = HeadingHierarchyOptions(enabled=True)

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(EMBED_MODEL),
    max_tokens=MAX_TOKENS,
)
chunker = HybridChunker(tokenizer=tokenizer, merge_peers=True)

## 2.2 Loading the PDFs (with cache)

`convert_with_cache` saves the Docling result for each PDF in `docling_cache/`, so the heavy conversion happens only once.
`load_docs` returns a `Document` for each PDF + the original `DoclingDocument`.

In [ ]:
def clean_id(name: str) -> str:
    return re.sub(r"\s+", " ", name).strip()


def convert_with_cache(pdf_path: Path) -> DoclingDocument:
    cache_file = CACHE_DIR / f"{clean_id(pdf_path.stem)}.json"
    if cache_file.exists():
        try:
            return DoclingDocument.load_from_json(cache_file)
        except Exception as e:
            print(f"Cache read failed for {pdf_path.name}, reconverting: {e}")
    dl_doc = converter.convert(str(pdf_path.resolve())).document
    try:
        dl_doc.save_as_json(cache_file)
    except Exception as e:
        print(f"Cache write failed for {pdf_path.name}: {e}")
    return dl_doc


def load_docs():
    """Returns (docs, dl_docs): one Document per PDF + the original DoclingDocument."""
    docs, dl_docs = [], {}
    for pdf_path in sorted(DATA_DIR.iterdir()):
        if not (pdf_path.is_file() and pdf_path.suffix.lower() == ".pdf"):
            continue
        doc_id = clean_id(pdf_path.stem)
        try:
            dl_doc = convert_with_cache(pdf_path)
            dl_docs[doc_id] = dl_doc
            docs.append(Document(
                page_content=dl_doc.export_to_markdown(),
                metadata={
                    "doc_id": doc_id,
                    "source": clean_id(pdf_path.name),
                    "num_pages": len(dl_doc.pages),
                },
            ))
            print(f"Successfully loaded: {pdf_path.name}")
        except Exception as e:
            print(f"Error loading {pdf_path.name}: {e}")
    return docs, dl_docs


def detect_first_page(dl_doc: DoclingDocument) -> int:
    """First page with a heading that marks the start of the actual content (otherwise 1)."""
    for item in dl_doc.texts:
        if isinstance(item, SectionHeaderItem) and CONTENT_START.search(item.text.strip()):
            if item.prov:
                return item.prov[0].page_no
    return 1

## 2.3 Chunking: `build_chunks`

Four stages: (1) read the Docling chunks and clean them (remove front/back matter and noise headings), (2) merge small chunks into the next one within the same section, (3) build a `Document` with rich metadata (headings, page range, has_table/code/formula), (4) IDs and `prev_id` / `next_id` links.

In [ ]:
def clean_headings(headings):
    return [h.strip() for h in (headings or []) if h.strip().lower() not in NOISE_HEADINGS]


def common_prefix(lists):
    out = []
    for items in zip(*lists):
        if all(i == items[0] for i in items):
            out.append(items[0])
        else:
            break
    return out


def build_chunks(doc_id, source, dl_doc, first_page=None):
    if first_page is None:
        first_page = FIRST_PAGE_OVERRIDE.get(doc_id) or detect_first_page(dl_doc)

    # 1) Read the original chunks and clean them
    units = []
    for ch in chunker.chunk(dl_doc=dl_doc):
        headings = clean_headings(ch.meta.headings)
        pages = {p.page_no for it in ch.meta.doc_items for p in it.prov}
        if pages and max(pages) < first_page:
            continue
        if NOISE_SECTIONS.search(" | ".join(headings)):
            continue
        units.append({
            "headings": headings,
            "text": ch.text,
            "pages": pages,
            "labels": {it.label.value for it in ch.meta.doc_items},
            "n": tokenizer.count_tokens(ch.text),
        })

    # 2) Merge small chunks into the next one (within the same main section)
    groups = []
    for u in units:
        g = groups[-1] if groups else None
        if (
            g
            and g["n"] < MIN_TOKENS
            and g["n"] + u["n"] <= HARD_MAX
            and g["items"][0]["headings"][:1] == u["headings"][:1]
        ):
            g["items"].append(u)
            g["n"] += u["n"]
        else:
            groups.append({"items": [u], "n": u["n"]})

    # 3) Build the Documents
    chunks = []
    for g in groups:
        if g["n"] < DROP_BELOW:
            continue
        common = common_prefix([u["headings"] for u in g["items"]])
        body = "\n\n".join(
            "\n".join(u["headings"][len(common):] + [u["text"]]) for u in g["items"]
        )
        content = ("\n".join(common) + "\n" + body) if common else body
        pages = sorted(set().union(*[u["pages"] for u in g["items"]]))
        labels = set().union(*[u["labels"] for u in g["items"]])

        chunks.append(Document(
            page_content=content,
            metadata={
                "doc_id": doc_id,
                "source": source,
                "headings": common,
                "section": common[-1] if common else None,
                "top_section": common[0] if common else None,
                "depth": len(common),
                "page_start": pages[0] if pages else None,
                "page_end": pages[-1] if pages else None,
                "has_table": DocItemLabel.TABLE.value in labels,
                "has_code": DocItemLabel.CODE.value in labels,
                "has_formula": DocItemLabel.FORMULA.value in labels,
                "n_tokens": tokenizer.count_tokens(content),
                "text_raw": body,
            },
        ))

    # 4) IDs and neighbor links
    for i, c in enumerate(chunks):
        c.metadata["chunk_id"] = f"{doc_id}::{i}"
        c.metadata["chunk_index"] = i
        c.metadata["prev_id"] = f"{doc_id}::{i-1}" if i else None
        c.metadata["next_id"] = f"{doc_id}::{i+1}" if i + 1 < len(chunks) else None
    return chunks

## 2.4 Embeddings and Qdrant

This cell builds the collection **if it doesn't exist** (or if `REBUILD = True`). If it exists, it skips Docling, chunking and embedding entirely, and only counts the chunks per document.

In [ ]:
from qdrant_client import QdrantClient, models
from langchain_huggingface import HuggingFaceEmbeddings

client = QdrantClient(url=QDRANT_URL)

embedder = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
    show_progress=False,
)


def build_table(client, chunks):
    vectors = embedder.embed_documents([c.page_content for c in chunks])
    client.create_collection(TABLE, vectors_config=models.VectorParams(size=DIM, distance=models.Distance.COSINE))
    client.create_payload_index(TABLE, "doc_id", models.PayloadSchemaType.KEYWORD)
    points = [models.PointStruct(id=str(uuid.uuid5(uuid.NAMESPACE_URL, c.metadata["chunk_id"])), vector=v,
                                 payload={**c.metadata, "content": c.page_content})
              for c, v in zip(chunks, vectors)]
    for i in tqdm(range(0, len(points), 64), desc="Uploading to Qdrant"):
        client.upload_points(TABLE, points[i:i + 64])


NEEDS_BUILD = REBUILD or not client.collection_exists(TABLE)

if NEEDS_BUILD:
    merged, dl_docs = load_docs()
    all_chunks = []
    for doc_id, dl_doc in dl_docs.items():
        fp = FIRST_PAGE_OVERRIDE.get(doc_id) or detect_first_page(dl_doc)
        doc_chunks = build_chunks(doc_id, f"{doc_id}.pdf", dl_doc, first_page=fp)
        print(f"{doc_id}: first_page={fp}, chunks={len(doc_chunks)}")
        all_chunks += doc_chunks
    print(len(all_chunks), all_chunks[0].metadata["chunk_id"])
    if client.collection_exists(TABLE):
        client.delete_collection(TABLE)
    build_table(client, all_chunks)
else:
    print(f"Collection '{TABLE}' already exists, skipping build (REBUILD=False)")

editions = (
    pd.DataFrame([p.payload for p in client.scroll(TABLE, limit=100_000, with_payload=["doc_id"], with_vectors=False)[0]])
    .groupby("doc_id").size().reset_index(name="all_chunks")
)
editions

# Part 3: `@step` (caching + time + tokens)

Every step in the pipeline is a small function, and `@step` wraps it and does two things:

- **Trace:** each call adds a line to the current question's trace: step name, input, seconds, tokens.
- **Cache (for steps with `cache=True`):** the result is saved in `experiments/steps.json` together with its time and tokens. Re-running the notebook reads the saved result instead of calling Groq again, and the same timings are shown.

LLM steps are cached. Local search and reranker steps are measured live and are not saved.

In [ ]:
from langchain_core.callbacks import get_usage_metadata_callback

STEPS = EXPERIMENTS / "steps.json"
saved = json.loads(STEPS.read_text(encoding="utf-8")) if STEPS.exists() else {}
current_trace = ContextVar("current_trace", default=None)
GPU = Lock()   # local models (embedder/reranker) run one at a time so their time is measured correctly


def step(name, cache=False, lock=None):
    def wrap(fn):
        memo = saved.setdefault(name, {}) if cache else None

        @wraps(fn)
        def run(*args):
            key = json.dumps(args, ensure_ascii=False)
            hit = memo.get(key) if cache else None
            if hit is None:
                with (lock or nullcontext()):
                    start = time.perf_counter()
                    with get_usage_metadata_callback() as usage:
                        result = fn(*args)
                    hit = {"result": result,
                           "seconds": round(time.perf_counter() - start, 3),
                           "tokens": sum(u.get("total_tokens", 0) for u in usage.usage_metadata.values())}
                if cache:
                    memo[key] = hit
            trace = current_trace.get()
            if trace is not None:
                trace.append({"step": name, "input": " ".join(str(args[0]).split())[:70] if args else "",
                              "seconds": hit["seconds"], "tokens": hit["tokens"]})
            return hit["result"]
        return run
    return wrap


def save():
    STEPS.write_text(json.dumps(saved, ensure_ascii=False), encoding="utf-8")

# Part 4: Search

## 4.1 Loading chunks from Qdrant and building BM25

All payloads are pulled once. `bm25_index(filter)` builds one index per filter, only once (stored with `@cache`). The filter is either `None` (all documents) or `("doc_id", "document name")`.
The BM25 tokenizer is `\w+` so Arabic text is not stripped.

In [ ]:
from rank_bm25 import BM25Okapi

points, offset = [], None
while True:
    batch, offset = client.scroll(TABLE, limit=500, offset=offset, with_payload=True, with_vectors=False)
    points += batch
    if offset is None:
        break
payloads = {str(p.id): p.payload for p in points}
ids = list(payloads)
DOC_IDS = sorted({p["doc_id"] for p in payloads.values()})

tok = lambda t: re.findall(r"\w+", t.lower())


def passes(payload, filter):
    return filter is None or payload[filter[0]] == filter[1]


@cache
def bm25_index(filter):
    sub = [i for i in ids if passes(payloads[i], filter)]
    return sub, BM25Okapi([tok(payloads[i]["content"]) for i in sub])


print(f"Loaded {len(ids)} chunks from {len(DOC_IDS)} documents")

## 4.2 vector / BM25 / hybrid

Each function takes an optional `filter`. Hybrid merges the two lists with RRF (each list adds `1 / (60 + rank)` to the chunk's score). `fuse` merges any number of lists, and `union` combines the results of sub-questions.

In [ ]:
def _qdrant_filter(filter):
    if filter is None:
        return None
    return models.Filter(must=[models.FieldCondition(key=filter[0], match=models.MatchValue(value=filter[1]))])


@step("vector", lock=GPU)
def dense_search(q, k=3, filter=None):
    res = client.query_points(TABLE, query=embedder.embed_query(q), limit=k, query_filter=_qdrant_filter(filter))
    return [(str(h.id), h.score) for h in res.points]


@step("bm25")
def bm25_search(q, k=3, filter=None):
    sub, bm = bm25_index(filter)
    if not sub:
        return []
    s = bm.get_scores(tok(q))
    top = sorted(range(len(sub)), key=lambda i: s[i], reverse=True)[:k]
    return [(sub[i], float(s[i])) for i in top]


def hybrid_search(q, k=3, filter=None, pool=50, rrf_k=60):
    fused = {}
    for res in (dense_search(q, pool, filter), bm25_search(q, pool, filter)):
        for rank, (pid, _) in enumerate(res, 1):
            fused[pid] = fused.get(pid, 0) + 1 / (rrf_k + rank)
    return sorted(fused.items(), key=lambda x: x[1], reverse=True)[:k]


def search(q, retriever, k, filter=None):
    fn = {"vector": dense_search, "bm25": bm25_search, "hybrid": hybrid_search}[retriever]
    return fn(q, k, filter)


def fuse(result_lists, k, rrf_k=60):
    scores = {}
    for res in result_lists:
        for rank, (pid, _) in enumerate(res, 1):
            scores[pid] = scores.get(pid, 0) + 1 / (rrf_k + rank)
    return sorted(scores, key=scores.get, reverse=True)[:k]


def union(found):
    return list(dict.fromkeys(pid for part in found for pid in part))

## 4.3 The Reranker

`cross_encoder(name)` loads the model once. `rerank` reads the question and the chunk together and re-orders them, and runs under a lock so its time is measured on its own.
The first run downloads bge (~2GB). `rerank_search` is used by the quick comparison and the refusal gate.

In [ ]:
from sentence_transformers import CrossEncoder

RERANKERS = {"bge": RERANK_MODEL, "minilm": "cross-encoder/ms-marco-MiniLM-L6-v2"}


@cache
def cross_encoder(name):
    return CrossEncoder(RERANKERS[name], max_length=1024 if name == "bge" else 512)


@step("rerank", lock=GPU)
def rerank(question, cand_ids, model, k):
    hits = cross_encoder(model).rank(question, [payloads[p]["content"] for p in cand_ids], show_progress_bar=False)
    return [cand_ids[h["corpus_id"]] for h in hits[:k]]


def rerank_search(q, k=3, pool=30):
    cands = [pid for pid, _ in hybrid_search(q, pool)]
    scores = cross_encoder("bge").predict([(q, payloads[p]["content"]) for p in cands])
    return sorted(zip(cands, map(float, scores)), key=lambda x: x[1], reverse=True)[:k]


print("bge on", cross_encoder("bge").device)

# Part 5: Quick comparison of the four methods

`compare` shows the results of Vector, BM25, Hybrid and Hybrid+Reranker side by side for a single question, and marks ✅/❌ if you set an `expect` word that should appear in the correct chunk.

In [ ]:
def show(name, res, expect=None):
    print(f"\n--- {name} ---")
    for r, (pid, s) in enumerate(res, 1):
        p = payloads[pid]
        ok = "" if not expect else (" ✅" if expect.lower() in p["content"].lower() else " ❌")
        print(f"#{r} | {s:.4f}{ok} | {p['chunk_id']} | p.{p['page_start']}-{p['page_end']} | {p.get('section')}")
        print("    " + p["content"][:250].replace("\n", " ") + "...")


def compare(query, k=3, expect=None):
    """expect: a word/phrase that should appear in the right chunk (optional, auto-marks ✅/❌)"""
    print(f"🔍 {query}\n" + "=" * 70)
    show("Vector", dense_search(query, k), expect)
    show("BM25", bm25_search(query, k), expect)
    show("Hybrid (RRF)", hybrid_search(query, k), expect)
    show("Hybrid + Reranker", rerank_search(query, k), expect)


compare("What is the ingestion flow in RAG?", k=3, expect="ingestion")

## 5.1 Manual question table (19 + one Arabic question)

For each question: the rank of the first chunk containing the `expect` word, for each method. Below the table: Hit@1 / Hit@3 / Hit@10 / MRR. The last Arabic question tests cross-lingual retrieval (BM25 is expected to fail on it).

In [ ]:
tests = [
    # --- Part one (production RAG: latency / caching) ---
    ("What is a good latency target for the retrieval components?", "300 ms"),
    ("How does the orchestrator query the vector DB and keyword search at the same time?", "fans out"),
    ("Why do we need an inference server like vLLM or TGI?", "continuous batching"),
    ("What are the layers where caching can be applied in a RAG pipeline?", "chunk cache"),
    ("How is semantic caching different from hash-based caching?", "similarity threshold"),
    ("How do I purge stale cache entries when a document is updated?", "Pub/Sub"),
    ("Which eviction policy is used when the cache runs out of RAM?", "least recently used"),
    ("How can I make a self-hosted vector database respond faster at scale?", "HNSW"),
    ("What does the improved prompt template look like?", "Helpful Answer"),
    # --- Part two (RAG-Driven Generative AI) ---
    ("What are the three main RAG configurations?", "modular rag"),
    ("What is the difference between parametric and non-parametric knowledge?", "non-parametric"),
    ("What does the trainer component do in the RAG ecosystem?", "trainer (T)"),
    ("Why should a RAG pipeline be split into independent components?", "parallel development"),
    ("What does cosine similarity measure?", "angle between two vectors"),
    ("Why does cosine similarity struggle with the vague query 'what is rag'?", "rag cloths"),
    ("How is index-based search different from plain vector search?", "indexed vector"),
    ("Which packages are needed for the data collection component?", "beautifulsoup4"),
    ("What happens in the Collect (D1) step of the retriever?", "PDFs filled with"),
    # --- Arabic question (tests cross-lingual) ---
    ("ايه الفرق بين RAG و fine-tuning ومتى أستخدم كل واحد؟", "fine-tuning"),
]

methods = {"Vector": dense_search, "BM25": bm25_search, "Hybrid": hybrid_search, "Hybrid+Rerank": rerank_search}


def rank_of(res, expect):
    for r, (pid, _) in enumerate(res, 1):
        if expect.lower() in payloads[pid]["content"].lower():
            return r
    return None


rows = []
for q, e in tests:
    row = {"question": q[:55], "expect": e}
    for name, fn in methods.items():
        row[name] = rank_of(fn(q, 10), e)
    rows.append(row)

df = pd.DataFrame(rows)
display(df.fillna("-"))

summary = {}
for name in methods:
    r = df[name]
    summary[name] = {"Hit@1": (r == 1).mean(), "Hit@3": (r <= 3).mean(), "Hit@10": r.notna().mean(),
                     "MRR": (1 / r).fillna(0).mean()}
display(pd.DataFrame(summary).T.round(2))

# Part 6: LLM steps

## 6.1 Groq and the prompts

The same prompts: rewrite, multi-query, decomposition and HyDE. If `GROQ_API_KEY` is not set, you will be asked for the key.
`max_retries=6` so any rate limit is retried automatically.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key: ")

chat = ChatGroq(model=LLM_MODEL, temperature=0, max_retries=6)


def instruction(text):
    return ChatPromptTemplate.from_messages([("system", text), ("human", "{question}")]) | chat


rewriter     = instruction("Rewrite this question as one short search query for a document search engine. Reply with the query only.")
multi_writer = instruction("Write 3 different versions of this question for a document search engine, one per line. Reply with the questions only.")
decomposer   = instruction("Split this question into separate standalone questions, one per line. If it asks only one thing, return it unchanged. Reply with the questions only.")
hyde_writer  = instruction("Write a short passage, as it might appear in a document, that answers this question. Reply with the passage only.")

print(chat.invoke("say hi").content)    # test: should reply

## 6.2 Pre-search steps (each is a cached `@step`)

`split`, `rewrite`, `variants` and `fake_answer`: each one is computed once per input and then saved.

In [ ]:
def lines(text):
    return [re.sub(r"^\s*(?:\d+[.)]|[-•*])\s*", "", l).strip() for l in text.splitlines() if l.strip()]


@step("decompose", cache=True)
def split(question):
    return lines(decomposer.invoke({"question": question}).content) or [question]


@step("rewrite", cache=True)
def rewrite(question):
    return rewriter.invoke({"question": question}).content.strip()


@step("multi_query", cache=True)
def variants(question):
    return lines(multi_writer.invoke({"question": question}).content)[:3]


@step("hyde", cache=True)
def fake_answer(question):
    return hyde_writer.invoke({"question": question}).content


test_q = "What is the ingestion flow in RAG and why is chunking important?"
print(split(test_q)); print(rewrite(test_q)); print(variants(test_q)); print(fake_answer(test_q))

## 6.3 Self-query: the LLM picks the target document
In the course, the planner chose the **version year** of a document. Here the documents are different books, so the planner reads the list of documents and answers with a document name **only if the question clearly asks for a specific document**; otherwise `NONE` and the search runs over all documents.
`to_filter` turns the name into a `("doc_id", name)` filter. This filter uses the payload index on `doc_id`, which was created but not used before.

In [ ]:
CATALOG = "\n".join(f"- {d}" for d in DOC_IDS)

planner = ChatPromptTemplate.from_messages([
    ("system", "You plan searches over a document store. It holds these documents:\n{catalog}\n\n"
               "If the question clearly asks about ONE specific document from the list (by its title or topic), "
               "reply with that document's exact name from the list. Otherwise reply NONE. Reply with the name only."),
    ("human", "{question}")]) | chat


@step("plan", cache=True)
def plan_doc(question):
    out = planner.invoke({"catalog": CATALOG, "question": question}).content.strip()
    return out if out in DOC_IDS else ""


def to_filter(doc):
    return ("doc_id", doc) if doc else None


for q in ["What does the cosine similarity section say about vague queries?", test_q]:
    print(q, "→", to_filter(plan_doc(q)))

## 6.4 The answer: `generate`

The same strict prompt: answer only from the numbered sources, with a citation after every sentence, and a fixed refusal if the sources don't contain the answer. It is cached as well.

In [ ]:
REFUSAL = "I don't know based on the documents."
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You answer questions strictly from the numbered sources below. "
               "Never use your own knowledge, even if you are sure of the answer. "
               "If the sources do not clearly contain the answer, reply with exactly this sentence and nothing else: "
               + REFUSAL + " "
               "Otherwise answer using only the sources, and after each sentence cite the sources it uses, like [1] or [2][3]."),
    ("human", "Sources:\n\n{context}\n\nQuestion: {question}")])


@step("generate", cache=True)
def generate(question, pids):
    sources = "\n\n".join(
        f"[{i}] {payloads[p]['doc_id']}, pages {payloads[p]['page_start']}-{payloads[p]['page_end']}\n{payloads[p]['content']}"
        for i, p in enumerate(pids, 1))
    return (answer_prompt | chat).invoke({"context": sources, "question": question}).content

# Part 7: `Config`, `retrieve` and `ask`

## 7.1 `Config`: one flag per technique

`Config` is frozen, so it can't be changed by accident, and `replace` creates a new one.
`self_query` is new: it plans a document filter for each part of the question.

In [ ]:
@dataclass(frozen=True)
class Config:
    name: str
    retriever: str = "vector"            # vector | bm25 | hybrid
    self_query: bool = False             # the LLM picks the document and filters on it
    query_steps: tuple[str, ...] = ()    # decompose, rewrite, multi_query, hyde
    rerank: str | None = None            # None | bge | minilm
    rerank_depth: int = 20               # how many candidates the reranker reads
    k: int = 4                           # how many chunks per sub-question go to the LLM

## 7.2 `retrieve` and `ask`

The order for each config:

1. **Decompose** (if enabled): the question is split into parts, otherwise a single part.
2. For each part: **self-query** (filter) → **rewrite** → **multi-query / HyDE** add queries → search each query → `fuse` → **rerank** (if enabled) or take the first `k`.
3. **Union** of the parts, then **generate**.

The reranker scores the chunks against the **original part**, not the rewritten query. `ask` returns the answer, the ids and the trace.

In [ ]:
def retrieve(question, config):
    parts = split(question) if "decompose" in config.query_steps else [question]
    found = []
    for part in parts:
        flt = to_filter(plan_doc(part)) if config.self_query else None
        query = rewrite(part) if "rewrite" in config.query_steps else part
        queries = [query]
        if "multi_query" in config.query_steps:
            queries += variants(query)
        if "hyde" in config.query_steps:
            queries.append(fake_answer(query))
        depth = config.rerank_depth if config.rerank else config.k
        candidates = fuse([search(q, config.retriever, depth, flt) for q in queries], depth)
        found.append(rerank(part, candidates, config.rerank, config.k) if config.rerank else candidates[:config.k])
    return union(found)


def ask(question, config):
    trace = []
    token = current_trace.set(trace)
    try:
        pids = retrieve(question, config)
        answer = generate(question, pids)
    finally:
        current_trace.reset(token)
    return {"answer": answer, "ids": pids, "trace": trace}

## 7.3 The ladder: 6 configs

Each stage turns on one more flag than the one before. The first is plain vector search, the last has everything enabled.

In [ ]:
BASELINE   = Config("baseline")
HYBRID     = replace(BASELINE, name="+hybrid", retriever="hybrid")
SELF_QUERY = replace(HYBRID, name="+self-query", self_query=True)
DECOMPOSE  = replace(SELF_QUERY, name="+decomposition", query_steps=("decompose",))
RERANK     = replace(DECOMPOSE, name="+reranker", rerank="bge")
EVERYTHING = replace(RERANK, name="everything", query_steps=("decompose", "rewrite", "multi_query", "hyde"))
HYBRID_RERANK = replace(HYBRID, name="+hybrid+rerank", rerank="bge")
LADDER = [BASELINE, HYBRID, SELF_QUERY, DECOMPOSE, RERANK, EVERYTHING, HYBRID_RERANK]
pd.DataFrame([vars(c) for c in LADDER]).set_index("name")

## 7.4 Trying a single question: `ask` with trace

You see the answer, the steps, and the time and tokens of each step.

In [ ]:
out = ask(test_q, RERANK)
for t in out["trace"]:
    print(f"{t['step']:12} {t['seconds']:6.2f} s {t['tokens']:6} tokens   {t['input']}")
print()
print(out["answer"])
save()

# Part 8: Golden set

The `GOLDEN_PATH` file must have the columns: `id`, `type`, `question`, `evidence`, `reference_answer`. A question whose `evidence` is empty is treated as a question that **should be refused**.
If the file has a `should_refuse` column it is used, otherwise it is computed from `evidence`. `||` separates multiple pieces of evidence in a compound question.

In [ ]:
SEP = "||"
golden = pd.read_csv(GOLDEN_PATH, keep_default_na=False)
if "should_refuse" not in golden.columns:
    golden["should_refuse"] = golden.evidence == ""
gq = golden.set_index("id")
answerable = golden[~golden.should_refuse]
print(len(golden), "سؤال،", len(answerable), "ليها إجابة،", int(golden.should_refuse.sum()), "المفروض يترفضوا")
golden.groupby("type").size().rename("questions").to_frame().T

## 8.1 `show_case`: one question in detail

The question, the trace (time and tokens of each step), the chunks (✅ = contains the evidence), the answer and the expected answer.

In [ ]:
def norm(t):
    return " ".join(t.lower().split())


def pieces(row):
    return row.evidence.split(SEP) if row.evidence else []


def holds(evidence, pid):
    return norm(evidence) in norm(payloads[pid]["content"])


def show_case(qid, config):
    row = gq.loc[qid]
    out = ask(row.question, config)
    print(f"{qid} ({row['type']}): {row.question}\nconfig: {config.name}\n")
    for t in out["trace"]:
        print(f"   {t['step']:11} {t['seconds']:5.2f} s {t['tokens']:5} tokens   {t['input']}")
    print(f"   {'total':11} {sum(t['seconds'] for t in out['trace']):5.2f} s\n")
    for rank, pid in enumerate(out["ids"], 1):
        p = payloads[pid]
        mark = "✅" if any(holds(e, pid) for e in pieces(row)) else "  "
        print(f"{mark} {rank}. {p['chunk_id']:30} p.{p['page_start']}-{p['page_end']}  {' '.join(p['content'].split())[:60]}")
    print(f"\nLLM:      {out['answer']}\nExpected: {row.reference_answer}")
    save()


show_case(golden.id.iloc[0], BASELINE)

# Part 9: Running the ladder on all questions

`run` asks every question with the config, `WORKERS` at a time, and saves every answer in `experiments/answers.jsonl` under the key (config + id + question text). This means:

- Re-running only computes new or edited questions.
- If a question fails (e.g. a rate limit), a warning is printed and it is skipped; it is computed on its own the next time you re-run.

`SAMPLE` is for a quick trial on the first N questions (None = all).

In [ ]:
ANSWERS = EXPERIMENTS / "answers.jsonl"
done = {}
if ANSWERS.exists():
    for line in ANSWERS.read_text(encoding="utf-8").splitlines():
        record = json.loads(line)
        done[record["key"]] = record

SAMPLE = None   # e.g. 10 for a quick trial


def key(config, row):
    return f"{config}|{row.id}|{row.question}"


def safe_ask(row, config):
    try:
        return ask(row.question, config)
    except Exception as err:
        print(f"⚠️ {config.name} {row.id}: {err}")
        return None


def run(config, rows=None):
    rows = golden if rows is None else rows
    todo = [r for r in rows.itertuples() if key(config, r) not in done]
    with ThreadPoolExecutor(WORKERS) as pool:
        outs = list(pool.map(lambda r: safe_ask(r, config), todo))
    with ANSWERS.open("a", encoding="utf-8") as f:
        for r, out in zip(todo, outs):
            if out is None:
                continue
            done[key(config, r)] = rec = {"key": key(config, r), "config": config.name, "id": r.id, **out}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    save()
    return [done[key(config, r)] for r in rows.itertuples() if key(config, r) in done]


rows_to_run = golden.head(SAMPLE) if SAMPLE else golden
outputs = {}
for config in LADDER:
    start = time.perf_counter()
    outputs[config.name] = run(config, rows_to_run)
    print(f"{config.name:15} {len(outputs[config.name]):3} answers  {time.perf_counter() - start:7.1f} s")

In [ ]:
import contextvars

ANSWERS = EXPERIMENTS / "answers.jsonl"
done = {}
if ANSWERS.exists():
    for line in ANSWERS.read_text(encoding="utf-8").splitlines():
        record = json.loads(line)
        done[record["key"]] = record

SAMPLE = None   # e.g. 10 for a quick trial


def key(config, row):
    return f"{config}|{row.id}|{row.question}"


def safe_ask(row, config):
    try:
        return ask(row.question, config)
    except Exception as err:
        print(f"⚠️ {config.name} {row.id}: {err!r}")
        return None


def run(config, rows=None):
    rows = golden if rows is None else rows
    todo = [r for r in rows.itertuples() if key(config, r) not in done]
    with ThreadPoolExecutor(WORKERS) as pool:
        # each question gets a copy of the kernel's context (including shell_parent)
        futures = [pool.submit(contextvars.copy_context().run, safe_ask, r, config) for r in todo]
        outs = [f.result() for f in futures]
    with ANSWERS.open("a", encoding="utf-8") as f:
        for r, out in zip(todo, outs):
            if out is None:
                continue
            done[key(config, r)] = rec = {"key": key(config, r), "config": config.name, "id": r.id, **out}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    save()
    return [done[key(config, r)] for r in rows.itertuples() if key(config, r) in done]


rows_to_run = golden.head(SAMPLE) if SAMPLE else golden
outputs = {}
for config in LADDER:
    start = time.perf_counter()
    outputs[config.name] = run(config, rows_to_run)
    print(f"{config.name:15} {len(outputs[config.name]):3} answers  {time.perf_counter() - start:7.1f} s")

# Part 10: Retrieval evaluation with ranx

We measure **exactly the chunks that went to the LLM** (4 per part, so up to 8 for a compound question). Each piece of evidence is one relevant item (`e0`, `e1`).

- **hit_rate:** did it find any evidence?  **recall:** how many of the evidence pieces did it find?
- **mrr / ndcg:** the rank of the evidence (the higher, the better).
- **complete:** the share of questions where it found **all** of their evidence.

In [ ]:
from ranx import Qrels, Run, evaluate as ranx_evaluate

METRICS = ["hit_rate", "recall", "mrr", "ndcg"]


def ranked(row, ids_):
    run_ = {}
    for rank, pid in enumerate(ids_, 1):
        hit = next((f"e{i}" for i, p in enumerate(pieces(row)) if f"e{i}" not in run_ and holds(p, pid)), pid)
        run_.setdefault(hit, 1 / rank)
    return run_


def found_all(row, ids_):
    return all(any(holds(p, pid) for pid in ids_) for p in pieces(row))


def retrieval_scores(outs, rows=None):
    got = {o["id"]: o["ids"] for o in outs}
    rows = answerable if rows is None else rows
    rows = rows[rows["id"].isin(got)]
    if rows.empty:
        return {}
    qrels = Qrels({r.id: {f"e{i}": 1 for i in range(len(pieces(r)))} for r in rows.itertuples()})
    run_ = Run({r.id: ranked(r, got[r.id]) for r in rows.itertuples()})
    scores = ranx_evaluate(qrels, run_, METRICS)
    return {**{m: float(scores[m]) for m in METRICS},
            "complete": sum(found_all(r, got[r.id]) for r in rows.itertuples()) / len(rows)}


retrieval = pd.DataFrame({name: retrieval_scores(outs) for name, outs in outputs.items()}).T.round(2)
retrieval

## 10.1 `complete` by question type

See which stage fixes which type (e.g. decomposition for compound questions, and the reranker for questions that are close to each other).

In [ ]:
complete_by_type = pd.DataFrame({
    name: {t: retrieval_scores(outs, rows).get("complete") for t, rows in answerable.groupby("type")}
    for name, outs in outputs.items()
}).round(2)
complete_by_type

# Part 11: Answer evaluation with Ragas (the judge is Groq)

Ragas uses Groq as the judge through the OpenAI-compatible endpoint. The metrics:
**faithfulness** (does the answer stick to the chunks?), **factual_correctness** (is it correct compared to the reference answer?), **context_precision** and **context_recall** (are the chunks relevant and complete?).
`answer_relevancy` was not added because it needs an extra embedding model for the judge.

**Important:** each question costs several judge calls, so choose in `CONFIGS_TO_SCORE` only the configs you need (e.g. hybrid and reranker) to avoid hitting the rate limit. Results are saved in `answers.jsonl`, so they are not recomputed.

In [ ]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextPrecision, ContextRecall, FactualCorrectness, Faithfulness

CONFIGS_TO_SCORE = ["+hybrid", "+reranker"]   # change to: ["baseline", "+hybrid", ..., "everything"]
RAGAS_CONCURRENCY = 2

aclient = AsyncOpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1")
judge = llm_factory(LLM_MODEL, client=aclient, reasoning_effort="low", max_tokens=4096)
if "max_tokens" in judge.model_args:
    judge.model_args["max_completion_tokens"] = judge.model_args.pop("max_tokens")

faithfulness, factual_correctness = Faithfulness(llm=judge), FactualCorrectness(llm=judge)
context_precision, context_recall = ContextPrecision(llm=judge), ContextRecall(llm=judge)
NAMES = ["faithfulness", "factual_correctness", "context_precision", "context_recall"]

## 11.1 Running the judge

Each answer is evaluated and its result is saved inside the same record in `answers.jsonl`, together with the reference answer it was evaluated against. If the reference answer changes in the golden set, the answer is re-evaluated. If evaluation fails for a question, it is skipped and retried next time.

In [ ]:
limit = asyncio.Semaphore(RAGAS_CONCURRENCY)
golden_rows = {r.id: r for r in golden.itertuples()}


async def score(row, record):
    contexts = [payloads[p]["content"] for p in record["ids"]]
    answer = record["answer"]
    async with limit:
        try:
            results = await asyncio.gather(
                faithfulness.ascore(user_input=row.question, response=answer, retrieved_contexts=contexts),
                factual_correctness.ascore(response=answer, reference=row.reference_answer),
                context_precision.ascore(user_input=row.question, reference=row.reference_answer, retrieved_contexts=contexts),
                context_recall.ascore(user_input=row.question, retrieved_contexts=contexts, reference=row.reference_answer))
        except Exception as err:
            print(f"⚠️ judge failed for {record['config']} {record['id']}: {err}")
            return
    record["scores"] = {name: result.value for name, result in zip(NAMES, results)}
    record["reference"] = row.reference_answer


def rewrite_answers():
    ANSWERS.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in done.values()), encoding="utf-8")


for name in CONFIGS_TO_SCORE:
    outs = outputs[name]
    todo = [(golden_rows[o["id"]], o) for o in outs if o.get("reference") != golden_rows[o["id"]].reference_answer]
    await asyncio.gather(*(score(row, out) for row, out in todo))
    rewrite_answers()
    print(f"{name:15} scored {len(todo)}")

## 11.2 Answers table

- **faithfulness:** average over the questions the bot answered (not refused).
- **context_precision / context_recall:** average over the questions that have an answer.
- **factual_correctness:** average over all questions (a correct refusal scores 1).

In [ ]:
def scored(outs):
    rows_ = [{"id": o["id"], "answer": o["answer"], **o["scores"]} for o in outs if "scores" in o]
    return golden.merge(pd.DataFrame(rows_), on="id")


def summarise(r):
    answered = r[~r.answer.str.contains(REFUSAL, case=False, regex=False)]
    has_answer = r[~r.should_refuse]
    return {"faithfulness": answered.faithfulness.mean(), "factual_correctness": r.factual_correctness.mean(),
            "context_precision": has_answer.context_precision.mean(), "context_recall": has_answer.context_recall.mean()}


results = {name: scored(outputs[name]) for name in CONFIGS_TO_SCORE}
answers = pd.DataFrame({name: summarise(r) for name, r in results.items()}).T.round(2)
answers

## 11.3 `factual_correctness` by question type

Read the rows before the averages: a single question can drag a whole type down (e.g. code or a short number that the judge can't split into claims).

In [ ]:
by_type = pd.DataFrame({name: r.groupby("type").factual_correctness.mean() for name, r in results.items()}).round(2)
by_type

# Part 12: Time and tokens per step, and the scorecard

## 12.1 Per step

Every answer comes with a trace. Here is the average **per question**: how many times each step ran, its seconds and its tokens. The steps of one question run one after another, so the question time = the sum of its steps.

In [ ]:
traces = pd.DataFrame([{"config": name, "id": o["id"], **t} for name, outs in outputs.items() for o in outs for t in o["trace"]])
n_q = pd.Series({name: len(outs) for name, outs in outputs.items()})

per_step = traces.groupby(["config", "step"]).agg(calls=("seconds", "size"), seconds=("seconds", "sum"), tokens=("tokens", "sum"))
per_step = per_step.div(per_step.index.get_level_values("config").map(n_q), axis=0).round(2)
per_step.loc["everything"].sort_values("seconds", ascending=False)

## 12.2 Per config

Average seconds, tokens, number of LLM calls and number of chunks sent to the LLM, **per question**.

In [ ]:
cost = traces.groupby("config").agg(seconds=("seconds", "sum"), tokens=("tokens", "sum"),
                                    llm_calls=("tokens", lambda t: (t > 0).sum()))
cost = cost.div(n_q, axis=0).reindex([c.name for c in LADDER])
cost["chunks"] = [sum(len(o["ids"]) for o in outputs[c.name]) / max(len(outputs[c.name]), 1) for c in LADDER]
cost.round(2)

## 12.3 The scorecard

Quality (ranx + Ragas) and cost (time and tokens) in one table, saved in `experiments/scorecard.csv`.
From it you decide which config to run: the one that gives you the highest `complete` and `factual_correctness` with the lowest time and tokens.

In [ ]:
scorecard = retrieval.join(cost.round(2))
if "answers" in globals():
    scorecard = retrieval.join(answers).join(cost.round(2))
scorecard.to_csv(EXPERIMENTS / "scorecard.csv")
scorecard

In [ ]:
BEST = HYBRID_RERANK   # change after reading the scorecard

# Part 13: Real usage

## 13.1 Refusal gate (custom)

If the best chunk's reranker score is below `GATE_MIN`, we answer "I don't know" without calling the LLM. This cell shows the lowest scores of real questions and the scores of off-topic questions, so you can choose the threshold.

In [ ]:
def relevance(question, n=3):
    cands = [pid for pid, _ in hybrid_search(question, n)]
    scores = cross_encoder("bge").predict([(question, payloads[p]["content"]) for p in cands])
    return round(float(max(scores)), 3)


good = sorted(relevance(q) for q in golden[~golden.should_refuse].question)
bad = {q: relevance(q) for q in ["What is the capital of France?",
                                 "How do I bake a chocolate cake?",
                                 "Who won the 2022 World Cup?"]}
print("أسئلة حقيقية، أقل ٥ درجات:", good[:5])
print("أسئلة بعيدة:", bad)

## 13.2 `answer_question` and `chat_ask` (custom)

`answer_question` goes through the gate, then `ask` with `BEST`, and returns the answer with only the sources that were actually cited (250-character excerpts). `chat_ask` prints it in a readable form.

In [ ]:
GATE_MIN = 0.1        # change to the number you chose from the cell above


def answer_question(question, config=None):
    config = config or BEST
    t0 = time.perf_counter()

    # gate: if no chunk is related to the question, answer "I don't know" without the LLM
    if relevance(question) < GATE_MIN:
        return {"question": question, "answer": REFUSAL, "sources": [],
                "seconds": round(time.perf_counter() - t0, 2)}

    out = ask(question, config)
    answer = out["answer"].strip()

    if REFUSAL.lower() in answer.lower():
        picked = []
    else:
        cited = sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer)})
        picked = [n for n in cited if 1 <= n <= len(out["ids"])] or list(range(1, len(out["ids"]) + 1))

    sources = []
    for n in picked:
        p = payloads[out["ids"][n - 1]]
        sources.append({"n": n, "doc_id": p["doc_id"], "pages": f"{p['page_start']}-{p['page_end']}",
                        "section": p.get("section"), "snippet": " ".join(p["content"].split())[:250]})
    return {"question": question, "answer": answer, "sources": sources,
            "seconds": round(time.perf_counter() - t0, 2)}


def chat_ask(question):
    r = answer_question(question)
    print(f"❓ {r['question']}\n\n💬 {r['answer']}\n")
    if r["sources"]:
        print("📚 المصادر:")
        for s in r["sources"]:
            print(f"  [{s['n']}] {s['doc_id']} | p.{s['pages']} | {s['section']}")
            print(f"      {s['snippet']}...")
    else:
        print("📚 مفيش مصادر")
    print(f"\n⏱️ {r['seconds']} s")

## 13.3 Try real questions and trap questions

A real question should be answered with sources, and the remaining questions (capital of France, cake, World Cup, EMNLP study, ada-002 price) should reply "I don't know" with no sources.

In [ ]:
chat_ask("What is the ingestion flow in RAG?")
chat_ask("What is the capital of France?")
chat_ask("How do I bake a chocolate cake?")
chat_ask("Who won the 2022 World Cup?")
chat_ask("How much does text-embedding-ada-002 cost per token?")
chat_ask("What did the 2024 EMNLP study find about semantic chunking?")

## 13.4 `find`: search for a word in all chunks

A debug tool: if a trap question was answered wrongly, search for the word in the original chunks and see whether it really exists in your documents.

In [ ]:
def find(keyword, width=150):
    for p in payloads.values():
        text = " ".join(p["content"].split())
        i = text.lower().find(keyword.lower())
        if i != -1:
            print(p["chunk_id"], "|", text[max(0, i - width): i + width], "\n")


find("capital of France")
find("Paris")